# 06 - Caso final: construir y justificar un pipeline

## Objetivo

Integrar esquemas, formatos, transformaciones, joins, particionado, planes y diagnóstico.
Primero se plantea el reto y después se incluye una solución de referencia.

In [ ]:
import os, shutil, subprocess

os.environ["JAVA_HOME"] = "/Users/beatrizsuch/java/temurin-17/Contents/Home"
os.environ["PATH"] = f'{os.environ["JAVA_HOME"]}/bin:' + os.environ["PATH"]

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("pontia-computacion-distribuida")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version} | {spark.sparkContext.master}")
print(f"Paralelismo por defecto: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

## 1. Generar las fuentes del caso

In [ ]:
from pyspark.sql.types import StructType, StructField, LongType, DoubleType

raw = (
    spark.range(0, 100_000, numPartitions=8)
    .withColumn("trip_id", F.col("id"))
    .withColumn("zone_id", (F.col("id") % 50).cast("long"))
    .withColumn("month", ((F.col("id") % 3) + 1).cast("int"))
    .withColumn(
        "fare",
        F.when(F.col("id") % 1_000 == 0, F.lit(-5.0))
         .otherwise(((F.col("id") % 60) + 1).cast("double")),
    )
    .select("trip_id", "zone_id", "month", "fare")
)

zones = spark.createDataFrame(
    [(i, f"zone_{i}", f"district_{i % 5}") for i in range(50)],
    ["zone_id", "zone_name", "district"],
)

raw_csv = DATA_DIR / "case_raw_csv"
zones_parquet = DATA_DIR / "case_zones_parquet"
raw.write.mode("overwrite").option("header", True).csv(str(raw_csv))
zones.write.mode("overwrite").parquet(str(zones_parquet))

## 2. Reto

Construye un pipeline que:

1. Lea el CSV con esquema explícito.
2. Elimine tarifas no positivas.
3. Lea la dimensión de zonas en Parquet.
4. Elija y justifique una estrategia de join.
5. Calcule viajes, tarifa media e ingresos por mes y distrito.
6. Escriba el resultado en Parquet particionado por mes.
7. Muestre el plan físico y señale los shuffles inevitables.
8. Compruebe el número de archivos generados.

Antes de consultar la solución, anota qué operadores esperas ver en `explain()`.

## 3. Solución de referencia: lectura y calidad

In [ ]:
input_schema = StructType([
    StructField("trip_id", LongType(), nullable=False),
    StructField("zone_id", LongType(), nullable=False),
    StructField("month", LongType(), nullable=False),
    StructField("fare", DoubleType(), nullable=True),
])

trips = (
    spark.read
    .option("header", True)
    .schema(input_schema)
    .csv(str(raw_csv))
    .filter(F.col("fare") > 0)
    .select("trip_id", "zone_id", "month", "fare")
)

zone_dimension = spark.read.parquet(str(zones_parquet))

## 4. Solución de referencia: join y agregación

In [ ]:
report = (
    trips
    .join(F.broadcast(zone_dimension), "zone_id", "inner")
    .groupBy("month", "district")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("fare"), 2).alias("avg_fare"),
        F.round(F.sum("fare"), 2).alias("revenue"),
    )
)

report.explain(mode="formatted")
report.orderBy("month", "district").show(20)

El broadcast evita redistribuir la tabla grande para el join. La agregación por mes y distrito
sí necesita reunir claves y mantiene un shuffle necesario.

## 5. Escritura consciente

In [ ]:
output_path = DATA_DIR / "case_report"
(
    report
    .repartition("month")
    .write
    .mode("overwrite")
    .partitionBy("month")
    .parquet(str(output_path))
)

part_files = list(output_path.rglob("part-*"))
print("Archivos de datos:", len(part_files))
for path in sorted(part_files):
    print(path.relative_to(output_path))

## 6. Validación del resultado escrito

In [ ]:
saved_report = spark.read.parquet(str(output_path))
february = saved_report.filter(F.col("month") == 2)
february.explain(mode="formatted")
february.orderBy("district").show()

## Discusión final

- ¿El `repartition("month")` previo a la escritura aporta valor o crea un shuffle redundante?
- ¿Cuántos archivos se han generado por mes y por qué?
- ¿Qué cambiaría si la dimensión dejara de ser pequeña?
- ¿Qué observarías en Spark UI antes de afirmar que el pipeline está optimizado?
- ¿Qué parte del razonamiento sería transferible a Flink, Trino u otro motor distribuido?

No existe una única solución óptima sin conocer volumen, recursos, frecuencia y patrón de uso.
La respuesta debe justificarse con el plan y con métricas.

In [ ]:
spark.stop()